In [1]:
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

# -------------------------
# SETTINGS
# -------------------------
num_employees = 172
num_claims = 7438

# -------------------------
# EMPLOYEES TABLE
# -------------------------
employee_ids = [f"E{str(i).zfill(4)}" for i in range(1, num_employees+1)]

employees = pd.DataFrame({
    "Employee_ID": employee_ids,
    "Age": np.random.randint(23, 57, num_employees),
    "Gender": np.random.choice(["Male","Female"], num_employees),
    "Marital_Status": np.random.choice(["Single","Married"], num_employees, p=[0.4,0.6]),
    "Number_of_Children": np.random.randint(0, 5, num_employees),
    "Department": np.random.choice(["Finance","HR","IT","Operations","Sales","Logistics"], num_employees),
    "Job_Level": np.random.choice(["Junior","Mid","Senior"], num_employees),
    "Salary": np.random.randint(2500, 8500, num_employees),
    "Hire_Date": [datetime(2015,1,1) + timedelta(days=random.randint(0, 3500)) for _ in range(num_employees)],
    "Company": "Company A",
    "Insurance_Plan": "SIC Corporate"
})

# -------------------------
# PROVIDERS TABLE
# -------------------------
providers_list = [
    ("City Pharmacy","Pharmacy"),
    ("Community Pharmacy","Pharmacy"),
    ("MedCare Pharmacy","Pharmacy"),
    ("General Hospital","Hospital"),
    ("Care Clinic","Hospital")
]

providers_df = pd.DataFrame(providers_list, columns=["Provider_Name","Facility_Type"])
overcharging_providers = ["MedCare Pharmacy"]

# -------------------------
# DRUGS TABLE
# -------------------------
drugs = [
    ("Paracetamol 500mg Tablet","Adult","Tablet",500),
    ("Paracetamol 120mg/5ml Syrup","Pediatric","Syrup",120),
    ("Ibuprofen 400mg Tablet","Adult","Tablet",400),
    ("Ibuprofen 100mg/5ml Suspension","Pediatric","Suspension",100),
    ("Amoxicillin 500mg Capsule","Adult","Capsule",500),
    ("Amoxicillin 125mg/5ml Suspension","Pediatric","Suspension",125),
    ("Ciprofloxacin 500mg Tablet","Adult","Tablet",500),
    ("Artemether-Lumefantrine Tablet","Adult","Tablet",80),
    ("Artemether-Lumefantrine Suspension","Pediatric","Suspension",20),
    ("Metformin 500mg Tablet","Adult","Tablet",500),
    ("Insulin Injection","Adult","Injection",100),
    ("Vitamin C 1000mg Effervescent","Adult","Effervescent",1000),
    ("Vitamin C Syrup","Pediatric","Syrup",100),
    ("Wellkid Multivitamin Syrup","Pediatric","Syrup",50),
    ("ORS Sachet","Pediatric","Powder",10),
    ("Zinc 20mg Tablet","Pediatric","Tablet",20),
    ("Dextromethorphan Syrup","Adult","Syrup",10),
    ("Diphenhydramine Syrup","Pediatric","Syrup",5)
]

drugs_df = pd.DataFrame(drugs, columns=["Drug_Name","Drug_Category","Formulation","Strength_mg"])

# -------------------------
# DIAGNOSIS MAP
# -------------------------
diagnosis_map = {
    "Malaria": ["Artemether-Lumefantrine Tablet","Artemether-Lumefantrine Suspension"],
    "Flu": ["Paracetamol 500mg Tablet","Paracetamol 120mg/5ml Syrup","Ibuprofen 400mg Tablet","Vitamin C 1000mg Effervescent","Vitamin C Syrup"],
    "Infection": ["Amoxicillin 500mg Capsule","Amoxicillin 125mg/5ml Suspension","Ciprofloxacin 500mg Tablet"],
    "Diabetes": ["Metformin 500mg Tablet","Insulin Injection"],
    "Hypertension": ["Metformin 500mg Tablet"],
    "General Wellness": ["Vitamin C 1000mg Effervescent","Vitamin C Syrup","Wellkid Multivitamin Syrup"]
}

diagnoses = list(diagnosis_map.keys())

# -------------------------
# ABUSERS
# -------------------------
abuser_ids = set(np.random.choice(employee_ids, size=14, replace=False))

# -------------------------
# CLAIMS GENERATION
# -------------------------
claims = []
start_date = datetime(2025,1,1)

for i in range(num_claims):
    emp = employees.sample(1).iloc[0]
    visit_date = start_date + timedelta(days=random.randint(0, 364))

    diagnosis = random.choice(diagnoses)
    drug_name = random.choice(diagnosis_map[diagnosis])
    drug_row = drugs_df[drugs_df["Drug_Name"] == drug_name].iloc[0]

    # Pricing
    if drug_row["Drug_Category"] == "Adult":
        base_price = random.uniform(18, 95)
    else:
        base_price = random.uniform(6, 40)

    # Pediatric misuse
    if emp["Age"] > 18 and random.random() < 0.06:
        ped = drugs_df[drugs_df["Drug_Category"] == "Pediatric"].sample(1).iloc[0]
        drug_row = ped

    provider = random.choice(providers_df["Provider_Name"].tolist())

    if provider in overcharging_providers:
        total_cost = base_price * random.uniform(1.8, 2.8)
    else:
        total_cost = base_price * random.uniform(0.9, 1.4)

    claims.append({
        "Claim_ID": f"C{str(i).zfill(6)}",
        "Employee_ID": emp["Employee_ID"],
        "Visit_Date": visit_date,
        "Month": visit_date.month,
        "Provider_Name": provider,
        "Diagnosis": diagnosis,
        "Drug_Name": drug_row["Drug_Name"],
        "Quantity": np.random.randint(1,4),
        "Unit_Price": round(base_price,2),
        "Total_Cost": round(total_cost,2),
        "Claim_Status": np.random.choice(["Approved","Rejected"], p=[0.93,0.07]),
        "Is_Abuser": 1 if emp["Employee_ID"] in abuser_ids else 0
    })

claims_df = pd.DataFrame(claims)

# -------------------------
# MERGED DATASET
# -------------------------
dataset = claims_df.merge(employees, on="Employee_ID", how="left")

# -------------------------
# SAVE TO EXCEL (MULTIPLE SHEETS)
# -------------------------
with pd.ExcelWriter("insurance_claims_2025_realistic.xlsx") as writer:
    dataset.to_excel(writer, sheet_name="Claims_Data", index=False)
    employees.to_excel(writer, sheet_name="Employees", index=False)
    claims_df.to_excel(writer, sheet_name="Claims_Only", index=False)
    drugs_df.to_excel(writer, sheet_name="Drugs", index=False)
    providers_df.to_excel(writer, sheet_name="Providers", index=False)

print("Excel file created with multiple sheets successfully!")  


"""

'\n\nimport pandas as pd\nimport numpy as np\nfrom datetime import datetime, timedelta\nimport random\n\nnp.random.seed(42)\nrandom.seed(42)\n\n# -------------------------\n# SETTINGS\n# -------------------------\nnum_employees = 172\nnum_claims = 7438\n\n# -------------------------\n# EMPLOYEES TABLE\n# -------------------------\nemployee_ids = [f"E{str(i).zfill(4)}" for i in range(1, num_employees+1)]\n\nemployees = pd.DataFrame({\n    "Employee_ID": employee_ids,\n    "Age": np.random.randint(23, 57, num_employees),\n    "Gender": np.random.choice(["Male","Female"], num_employees),\n    "Marital_Status": np.random.choice(["Single","Married"], num_employees, p=[0.4,0.6]),\n    "Number_of_Children": np.random.randint(0, 5, num_employees),\n    "Department": np.random.choice(["Finance","HR","IT","Operations","Sales","Logistics"], num_employees),\n    "Job_Level": np.random.choice(["Junior","Mid","Senior"], num_employees),\n    "Salary": np.random.randint(2500, 8500, num_employees),\n  

In [6]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

# -------------------------
# SETTINGS
# -------------------------
num_employees = 172
num_claims = 10237  # non-rounded realistic number

# -------------------------
# EMPLOYEES
# -------------------------
employee_ids = [f"E{str(i).zfill(4)}" for i in range(1, num_employees+1)]

employees = pd.DataFrame({
    "Employee_ID": employee_ids,
    "Age": np.random.randint(23, 57, num_employees),
    "Gender": np.random.choice(["Male","Female"], num_employees),
    "Marital_Status": np.random.choice(["Single","Married"], num_employees, p=[0.4,0.6]),
    "Number_of_Children": np.random.randint(0, 5, num_employees),
    "Department": np.random.choice(["Finance","HR","IT","Operations","Sales","Logistics"], num_employees),
    "Job_Level": np.random.choice(["Junior","Mid","Senior"], num_employees),
    "Salary": np.random.randint(2500, 8500, num_employees),
    "Hire_Date": [datetime(2015,1,1) + timedelta(days=random.randint(0, 3500)) for _ in range(num_employees)],
    "Company": "Concentrix Ghana",
    "Insurance_Plan": "GLICO Corporate"
})

# -------------------------
# PROVIDERS (REAL GHANA MIX)
# -------------------------
providers = [
    # Pharmacies
    ("Ernest Chemist", "Pharmacy"),
    ("Top-Up Pharmacy", "Pharmacy"),
    ("Dosty Pharmacy", "Pharmacy"),
    ("Unicom Chemist", "Pharmacy"),

    # Hospitals
    ("UGMC", "Hospital"),
    ("Korle-Bu Teaching Hospital", "Hospital"),
    ("37 Military Hospital", "Hospital"),
    ("Maritime Hospital", "Hospital"),
    ("Ridge Hospital", "Hospital"),
    ("Nyaho Medical Centre", "Hospital"),
    ("Lister Hospital", "Hospital"),
    ("Trust Hospital", "Hospital")
]

providers_df = pd.DataFrame(providers, columns=["Provider_Name","Facility_Type"])

# Overcharging providers (subtle)
overcharging_providers = ["Nyaho Medical Centre", "Lister Hospital"]

# -------------------------
# DRUGS
# -------------------------
drugs = [
    ("Paracetamol 500mg Tablet","Adult"),
    ("Paracetamol Syrup","Pediatric"),
    ("Ibuprofen","Adult"),
    ("Amoxicillin","Adult"),
    ("Ciprofloxacin","Adult"),
    ("Artemether-Lumefantrine","Adult"),
    ("Metformin","Adult"),
    ("Insulin","Adult"),
    ("Vitamin C Effervescent","Adult"),
    ("Vitamin C Syrup","Pediatric"),
    ("Wellkid Syrup","Pediatric"),
]

drugs_df = pd.DataFrame(drugs, columns=["Drug_Name","Drug_Category"])

# -------------------------
# DIAGNOSIS
# -------------------------
diagnoses = ["Malaria","Flu","Infection","Diabetes","Hypertension","General Wellness"]

# -------------------------
# CLAIMS
# -------------------------
claims = []
start_date = datetime(2025,1,1)

for i in range(num_claims):
    emp = employees.sample(1).iloc[0]
    visit_date = start_date + timedelta(days=random.randint(0, 364))

    provider_row = providers_df.sample(1).iloc[0]
    provider = provider_row["Provider_Name"]
    facility = provider_row["Facility_Type"]

    diagnosis = random.choice(diagnoses)
    drug_row = drugs_df.sample(1).iloc[0]

    quantity = np.random.randint(1,4)

    # -------------------------
    # PRICING LOGIC
    # -------------------------

    if facility == "Pharmacy":
        unit_price = random.uniform(25, 120)
        base_cost = unit_price * quantity

        if provider in overcharging_providers:
            total_cost = base_cost * random.uniform(1.2, 1.8)
        else:
            total_cost = base_cost * random.uniform(1.0, 1.3)

    else:  # Hospital
        admission_flag = np.random.choice([0,1], p=[0.75,0.25])

        if admission_flag == 0:
            # OPD
            total_cost = random.uniform(300, 900)
        else:
            # Admission (IVs, labs, bed, etc.)
            total_cost = random.uniform(1000, 3500)

        # slight inflation for private hospitals
        if provider in overcharging_providers:
            total_cost *= random.uniform(1.2, 1.5)

        unit_price = total_cost / quantity  # derived

    claims.append({
        "Claim_ID": f"C{str(i).zfill(6)}",
        "Employee_ID": emp["Employee_ID"],
        "Visit_Date": visit_date,
        "Provider_Name": provider,
        "Facility_Type": facility,
        "Diagnosis": diagnosis,
        "Drug_Name": drug_row["Drug_Name"],
        "Drug_Category": drug_row["Drug_Category"],
        "Quantity": quantity,
        "Unit_Price": round(unit_price,2),
        "Total_Cost": round(total_cost,2),
        "Admission": admission_flag if facility=="Hospital" else 0
    })

claims_df = pd.DataFrame(claims)

# -------------------------
# SAVE TO EXCEL
# -------------------------
with pd.ExcelWriter("ghana_realistic_insurance_dataset.xlsx") as writer:
    claims_df.to_excel(writer, sheet_name="Claims", index=False)
    employees.to_excel(writer, sheet_name="Employees", index=False)
    providers_df.to_excel(writer, sheet_name="Providers", index=False)
    drugs_df.to_excel(writer, sheet_name="Drugs", index=False)

print("🔥 Realistic Ghana insurance dataset generated.")

🔥 Realistic Ghana insurance dataset generated.
